# Dan's Comedy Clip Studio — free test notebook
This runs inside the normal Google Colab notebook interface. Upload your ~90 MB stand-up video from your phone, transcribe it, review suggested moments, then download a ZIP of vertical captioned clips.

Free Colab compute is temporary and GPU access is not guaranteed. Keep the notebook open until your clips are downloaded. Your video and exports live only in the temporary runtime unless you download them.

Run the cells from top to bottom.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install faster-whisper==1.1.1 pandas

## 1. Upload your video
Run the cell below and choose your stand-up video from your Android phone's files.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, os, re, subprocess, zipfile
import pandas as pd
from faster_whisper import WhisperModel
from IPython.display import display

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No video selected")
video_name = next(iter(uploaded))
VIDEO = Path("/content") / video_name
print(f"Uploaded {video_name} ({VIDEO.stat().st_size/1024/1024:.1f} MB)")

## 2. Transcribe and find candidate moments
This uses Whisper small. It uses a free GPU if Colab assigns one; otherwise it runs on CPU and may take longer. Candidate detection is a rough first pass, not an automatic comedy judge.

In [ ]:
import torch
has_gpu = torch.cuda.is_available()
print("GPU available:", has_gpu)
model = WhisperModel("small", device="cuda" if has_gpu else "cpu", compute_type="float16" if has_gpu else "int8")
segments_iter, info = model.transcribe(str(VIDEO), word_timestamps=False, vad_filter=True)
SEGMENTS = [{"start":float(s.start),"end":float(s.end),"text":s.text.strip()} for s in segments_iter]
duration = float(info.duration or SEGMENTS[-1]["end"])
print(f"Transcribed {len(SEGMENTS)} speech segments; video duration {duration/60:.1f} minutes")

cue = re.compile(r"\b(why|because|literally|apparently|obviously|ridiculous|imagine|sorry|fuck|shit|hell|what)\b", re.I)
ranked = []
for i,s in enumerate(SEGMENTS):
    score = 2*len(cue.findall(s["text"])) + min(len(s["text"])/90, 2)
    if i+1 < len(SEGMENTS):
        gap = SEGMENTS[i+1]["start"] - s["end"]
        if 0.35 < gap < 2.8:
            score += 1.2
    if score >= 1.5:
        a=max(0,s["start"]-9); b=min(duration,s["end"]+7)
        if b-a >= 8:
            ranked.append({"start":round(a,2),"end":round(b,2),"title":s["text"][:100],"score":round(score,2)})
ranked.sort(key=lambda x:x["score"], reverse=True)
CANDIDATES=[]
for c in ranked:
    if all(c["end"]<=x["start"] or c["start"]>=x["end"] for x in CANDIDATES):
        CANDIDATES.append(c)
    if len(CANDIDATES)==20:
        break
CANDIDATES.sort(key=lambda x:x["start"])
display(pd.DataFrame(CANDIDATES))
print("Full transcript:")
print("\n".join(f'[{s["start"]:.1f}s] {s["text"]}' for s in SEGMENTS))

## 3. Choose moments and adjust timings
Look at the candidate table above. Change the numbers in SELECTED to the candidate row numbers you want (starting at 0). You can also adjust start/end seconds in TIMING_OVERRIDES, e.g. {0: (32.0, 57.0)}.

In [ ]:
# EDIT THESE TWO LINES
SELECTED = [0, 1]  # candidate row numbers to export, e.g. [0, 3, 5]
TIMING_OVERRIDES = {}  # optional: {candidate_row: (start_seconds, end_seconds)}

chosen=[]
for idx in SELECTED:
    if idx<0 or idx>=len(CANDIDATES):
        print("Skipping invalid candidate row",idx); continue
    c=CANDIDATES[idx].copy()
    if idx in TIMING_OVERRIDES:
        c["start"],c["end"]=TIMING_OVERRIDES[idx]
    if c["end"]<=c["start"] or c["start"]<0:
        print("Skipping invalid timing",idx); continue
    chosen.append(c)
display(pd.DataFrame(chosen))
print("Selected",len(chosen),"clip(s). If correct, run the export cell below.")

## 4. Export vertical clips with subtitles
The next cell creates 1080×1920 MP4 clips and bundles them into a ZIP. A download prompt should appear when it finishes.

In [ ]:
from pathlib import Path
OUT=Path("/content/comedy_clips"); OUT.mkdir(exist_ok=True)
def srt_time(t):
    ms=int(t*1000); h,ms=divmod(ms,3600000); m,ms=divmod(ms,60000); sec,ms=divmod(ms,1000)
    return f"{h:02}:{m:02}:{sec:02},{ms:03}"
exports=[]
for n,c in enumerate(chosen,1):
    start,end=float(c["start"]),float(c["end"])
    srt=OUT/f"clip-{n:02}.srt"; mp4=OUT/f"clip-{n:02}.mp4"
    entries=[]
    for seg in SEGMENTS:
        a=max(start,seg["start"]); b=min(end,seg["end"])
        if b>a:
            entries.append(f"{len(entries)+1}\n{srt_time(a-start)} --> {srt_time(b-start)}\n{seg['text']}\n")
    srt.write_text("\n".join(entries),encoding="utf-8")
    sub=str(srt).replace("\\","/").replace(":","\\:")
    vf=f"scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,subtitles='{sub}':force_style='FontSize=18,Outline=2,Shadow=1,Alignment=2,MarginV=150'"
    cmd=["ffmpeg","-y","-ss",str(start),"-i",str(VIDEO),"-t",str(end-start),"-vf",vf,"-c:v","libx264","-preset","veryfast","-crf","23","-c:a","aac","-b:a","128k",str(mp4)]
    p=subprocess.run(cmd,capture_output=True,text=True)
    if p.returncode:
        raise RuntimeError(p.stderr[-2000:])
    exports.append(mp4)
print("Exported:",[p.name for p in exports])
ZIP="/content/dans_comedy_clips.zip"
with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in exports:
        z.write(p,p.name)
from google.colab import files
files.download(ZIP)

## Finished
Download the ZIP to your phone before closing the notebook. If Colab disconnects, rerun the notebook and upload the original video again.